# Dev Notebook

In [7]:
from pydantic import Field
from pydantic.dataclasses import dataclass
from datetime import datetime, timezone
import os
import requests
from requests.auth import HTTPBasicAuth
from requests.exceptions import HTTPError
from dotenv import load_dotenv
from typing import Union, Dict, Any, ClassVar, List
import pandas as pd

from stashies.dataclasses import YarnCompany, YarnFiber, Yarn


In [8]:
def post_request(
    url: str,
    data: Union[Dict[str, Any], None] = None,
    json: Union[Dict[str, Any], None] = None,
    API_KEY: str = "cbb3172b2f9543fd000cffe8545ee810",
    PASSWORD: str = "_NdUPofaVn3vjIjInKvLIwVxj2tg4Dk5Momswj1a",
) -> Any:
    """
    Function for making POST requests to Ravelry API.
    - Input
        - url: full API endpoint URL
        - data: form data to send (optional)
        - json: JSON data to send (optional)
    - output: Data returned from API
    """
    try:
        response = requests.post(
            url, 
            auth=HTTPBasicAuth(API_KEY, PASSWORD),
            data=data,
            json=json
        )
        response.raise_for_status()
    except HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
    except Exception as err:
        print(f"Other error occurred: {err}")
    else:
        return response.json()
    
def get_request(
    url: str,
    API_KEY: str = "cbb3172b2f9543fd000cffe8545ee810",
    PASSWORD: str = "_NdUPofaVn3vjIjInKvLIwVxj2tg4Dk5Momswj1a",
) -> Any:
    """
    Function for making requests to Ravelry API.
    - Input
        - endpoint: end of API query e.x.`current_user.json`
    - output: Data returned from API
    """
    try:
        response = requests.get(url, auth=HTTPBasicAuth(API_KEY, PASSWORD))
        response.raise_for_status()
    except HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
    except Exception as err:
        print(f"Other error occurred: {err}")
    else:
        # self.LOGGER.info(response.json())
        return response.json()


In [9]:
USERNAME: str = "thotsky"  # "KMLadybugCrochets"

# USERNAME: str = "KMLadybugCrochets"


In [10]:
stash = get_request(url=f"https://api.ravelry.com/people/{USERNAME}/stash/list.json")[
    "stash"
]
stash_ids = [entry["id"] for entry in stash]
stash_id = stash_ids[0]

# post_request(
#     f"https://api.ravelry.com/people/{USERNAME}/stash/{stash_id}.json",
#     data={"skeins": 6.5},
# )
item = get_request(
    url=f"https://api.ravelry.com/people/{USERNAME}/stash/{stash_id}.json"
)["stash"]
del item['yarn']['photos']
del item['yarn']['notes_html']

yarn = Yarn(**item['yarn'])

item


{'comments_count': 0,
 'created_at': '2025/09/12 02:00:29 -0400',
 'dye_lot': None,
 'favorites_count': 0,
 'handspun': False,
 'has_photo': False,
 'id': 31516215,
 'location': None,
 'permalink': 'moonshine',
 'updated_at': '2025/09/12 02:00:29 -0400',
 'user_id': 13618756,
 'name': 'Juniper Moon Farm Moonshine',
 'notes': None,
 'notes_html': None,
 'stash_status': {'id': 1, 'name': 'In stash'},
 'colorway_name': 'Purple',
 'color_family_name': None,
 'yarn_weight_name': 'Worsted',
 'long_yarn_weight_name': 'Worsted (9 wpi)',
 'personal_yarn_weight': None,
 'tag_names': [],
 'photos': [],
 'yarn': {'discontinued': False,
  'gauge_divisor': 1,
  'grams': 100,
  'id': 102743,
  'machine_washable': None,
  'max_gauge': 5.0,
  'min_gauge': 4.5,
  'name': 'Moonshine',
  'permalink': 'juniper-moon-farm-moonshine',
  'rating_average': 4.69,
  'rating_count': 1735,
  'rating_total': 8129,
  'texture': 'plied',
  'thread_size': None,
  'wpi': None,
  'yardage': 197,
  'min_needle_size': {'id

In [11]:
# 
packs = item['packs']
packs[0]['timestamp'] = datetime.strptime(item['created_at'], '%Y/%m/%d %H:%M:%S %z')
current_time = datetime.now(timezone.utc)
current_time = current_time.strftime('%Y/%m/%d %H:%M:%S %z')



df = pd.DataFrame(packs)
df


,id,primary_pack_id,project_id,skeins,stash_id,total_grams,total_meters,total_ounces,total_yards,yarn_id,...,meters_per_skein,ounces_per_skein,prefer_metric_weight,prefer_metric_length,shop_id,thread_size,color_attributes,total_paid,total_paid_currency,timestamp
0,134290871,NaN,None,7.0,31516215,700,1261.0,24.69,1379.0,102743,...,180.1,3.53,True,False,None,None,[],None,None,2025-09-12 02:00:29-04:00
1,134290872,134290871.0,None,5.5,31516215,550,990.9,19.40,1083.5,102743,...,NaN,NaN,True,False,None,None,[],None,None,NaT
2,135830458,134290871.0,None,1.0,31516215,100,180.1,3.53,197.0,102743,...,180.1,3.53,True,False,None,None,[],None,None,NaT
3,135882039,134290871.0,None,0.5,31516215,50,90.1,1.76,98.5,102743,...,180.1,3.53,True,False,None,None,[],None,None,NaT


In [12]:
df[['total_grams', 'total_meters', 'total_ounces', 'total_yards', 'timestamp']]


,total_grams,total_meters,total_ounces,total_yards,timestamp
0,700,1261.0,24.69,1379.0,2025-09-12 02:00:29-04:00
1,550,990.9,19.40,1083.5,NaT
2,100,180.1,3.53,197.0,NaT
3,50,90.1,1.76,98.5,NaT
